In [1]:
# Parameters
Month = 2
Year = 2569


In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import os, time, shutil

# --- 1. ตั้งค่าพื้นฐาน ---
# (ตัวแปร Month, Year ต้องถูกส่งมาจาก Main Script หรือ Uncomment ด้านบน)
final_dir_month = os.path.join(os.getcwd(), "Data_U_Month")
final_dir_week = os.path.join(os.getcwd(), "Data_U_Week")
download_dir = os.getcwd()

for d in [final_dir_month, final_dir_week]:
    if not os.path.exists(d): os.makedirs(d)

# --- 2. Chrome Configuration ---
chrome_options = webdriver.ChromeOptions()
prefs = {"download.default_directory": download_dir}
chrome_options.add_experimental_option("prefs", prefs)
# chrome_options.add_argument("--headless") # เปิดถ้าไม่อยากให้หน้าต่างเด้งขึ้นมา

driver = webdriver.Chrome(options=chrome_options)

try:
    # --- 3. Login & Navigate ---
    driver.get("https://sso-tpso.moc.go.th/Login")
    driver.find_element(By.ID, "Username").send_keys("tanaponp")
    driver.find_element(By.ID, "Password").send_keys("&5#t2B3&" + Keys.RETURN)
    time.sleep(3)

    driver.get("https://dev-tpso.moc.go.th/export12c/cpi")
    time.sleep(2)

    # เลือกเดือน/ปี
    driver.execute_script(f"document.getElementById('ddlMonth').value = '{Month}';")
    driver.execute_script(f"document.getElementById('ddlYear').value = '{Year}';")

    # --- 4. สั่งดาวน์โหลดทั้ง 2 ไฟล์ ---
    targets = [
        {"btn_id": "btnCpiuMonth", "prefix": "cpiu_month", "dest": final_dir_month},
        {"btn_id": "btnCpiuWeek", "prefix": "cpiu_week", "dest": final_dir_week}
    ]

    for target in targets:
        driver.find_element(By.ID, target["btn_id"]).click()
        print(f"📥 คลิกดาวน์โหลด: {target['prefix']}")
        
        # รอจนไฟล์ปรากฏและโหลดเสร็จ (.crdownload หายไป)
        success = False
        for _ in range(30): # Timeout 60 วินาที
            files = [f for f in os.listdir(download_dir) if f.startswith(target["prefix"]) and f.endswith(".xlsx")]
            temp_files = [f for f in os.listdir(download_dir) if f.endswith(".crdownload")]
            
            if files and not temp_files:
                # ย้ายไฟล์ทันที
                shutil.move(os.path.join(download_dir, files[0]), os.path.join(target["dest"], files[0]))
                print(f"✅ สำเร็จ: {files[0]}")
                success = True
                break
            time.sleep(2)
        
        if not success:
            print(f"❌ ล้มเหลว: ไม่พบไฟล์ {target['prefix']} ในเวลาที่กำหนด")

finally:
    driver.quit()
    print("🏁 ปิดระบบการดาวน์โหลด")

📥 คลิกดาวน์โหลด: cpiu_month


✅ สำเร็จ: cpiu_month_2569-2.xlsx


📥 คลิกดาวน์โหลด: cpiu_week


✅ สำเร็จ: cpiu_week_2569-2.xlsx


🏁 ปิดระบบการดาวน์โหลด
